In [0]:
# Homework #5: Model Deployment
# Exercise #4
# Author: Tianyu Jiang

DATABASE_NAME = "gr5069.tj2611"
EXPERIMENT_NAME = "/Users/tj2611@columbia.edu/homework_5_model_deployment_iris"
MLFLOW_DFS_TMP = "/Volumes/gr5069/tj2611/inclass/mlflow_tmp"

In [0]:
spark.sql(f"USE {DATABASE_NAME}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DATABASE_NAME}.logistic_regression_iris_predictions_hw5 (
    flower_id INT,
    sepal_length DOUBLE,
    sepal_width DOUBLE,
    petal_length DOUBLE,
    petal_width DOUBLE,
    species STRING,
    target INT,
    prediction DOUBLE,
    probability STRING,
    model_name STRING,
    run_id STRING,
    prediction_timestamp TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DATABASE_NAME}.random_forest_iris_predictions_hw5 (
    flower_id INT,
    sepal_length DOUBLE,
    sepal_width DOUBLE,
    petal_length DOUBLE,
    petal_width DOUBLE,
    species STRING,
    target INT,
    prediction DOUBLE,
    probability STRING,
    model_name STRING,
    run_id STRING,
    prediction_timestamp TIMESTAMP
)
USING DELTA
""")

display(spark.sql(f"SHOW TABLES IN {DATABASE_NAME}"))

database,tableName,isTemporary
tj2611,logistic_regression_iris_predictions_hw5,false
tj2611,logistic_regression_predictions_hw5,false
tj2611,random_forest_iris_predictions_hw5,false
tj2611,random_forest_predictions_hw5,false


In [0]:
from pyspark.sql.functions import col, when
from pyspark.sql.types import IntegerType, DoubleType

iris_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/databricks-datasets/Rdatasets/data-001/csv/datasets/iris.csv")
)

display(iris_raw.limit(5))
print(iris_raw.columns)

_c0,Sepal.Length,Sepal.Width,Petal.Length,Petal.Width,Species
1,5.1,3.5,1.4,0.2,setosa
2,4.9,3.0,1.4,0.2,setosa
3,4.7,3.2,1.3,0.2,setosa
4,4.6,3.1,1.5,0.2,setosa
5,5.0,3.6,1.4,0.2,setosa


['_c0', 'Sepal.Length', 'Sepal.Width', 'Petal.Length', 'Petal.Width', 'Species']


In [0]:
from pyspark.sql.functions import col, when
from pyspark.sql.types import IntegerType, DoubleType

iris_df = (
    iris_raw
    .withColumnRenamed("_c0", "flower_id")
    .withColumnRenamed("Sepal.Length", "sepal_length")
    .withColumnRenamed("Sepal.Width", "sepal_width")
    .withColumnRenamed("Petal.Length", "petal_length")
    .withColumnRenamed("Petal.Width", "petal_width")
    .withColumnRenamed("Species", "species")
)

model_df = (
    iris_df
    .select(
        col("flower_id").cast(IntegerType()).alias("flower_id"),
        col("sepal_length").cast(DoubleType()).alias("sepal_length"),
        col("sepal_width").cast(DoubleType()).alias("sepal_width"),
        col("petal_length").cast(DoubleType()).alias("petal_length"),
        col("petal_width").cast(DoubleType()).alias("petal_width"),
        col("species")
    )
    .dropna()
)

model_df = model_df.withColumn(
    "target",
    when(col("species") == "setosa", 1).otherwise(0)
)

display(model_df)

flower_id,sepal_length,sepal_width,petal_length,petal_width,species,target
1,5.1,3.5,1.4,0.2,setosa,1
2,4.9,3.0,1.4,0.2,setosa,1
3,4.7,3.2,1.3,0.2,setosa,1
4,4.6,3.1,1.5,0.2,setosa,1
5,5.0,3.6,1.4,0.2,setosa,1
6,5.4,3.9,1.7,0.4,setosa,1
7,4.6,3.4,1.4,0.3,setosa,1
8,5.0,3.4,1.5,0.2,setosa,1
9,4.4,2.9,1.4,0.2,setosa,1
10,4.9,3.1,1.5,0.1,setosa,1


In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

feature_cols = [
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="raw_features"
)

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=True
)

train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

Training rows: 121
Testing rows: 29


In [0]:
import mlflow
import mlflow.spark
import pandas as pd
import tempfile
import os

from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import current_timestamp, lit, col

mlflow.set_experiment(EXPERIMENT_NAME)

def evaluate_predictions(predictions):
    auc_evaluator = BinaryClassificationEvaluator(
        labelCol="target",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )

    accuracy_evaluator = MulticlassClassificationEvaluator(
        labelCol="target",
        predictionCol="prediction",
        metricName="accuracy"
    )

    f1_evaluator = MulticlassClassificationEvaluator(
        labelCol="target",
        predictionCol="prediction",
        metricName="f1"
    )

    precision_evaluator = MulticlassClassificationEvaluator(
        labelCol="target",
        predictionCol="prediction",
        metricName="weightedPrecision"
    )

    return {
        "auc": auc_evaluator.evaluate(predictions),
        "accuracy": accuracy_evaluator.evaluate(predictions),
        "f1": f1_evaluator.evaluate(predictions),
        "weighted_precision": precision_evaluator.evaluate(predictions)
    }


def log_two_artifacts(predictions, model_name):
    prediction_pdf = predictions.select(
        "flower_id",
        "species",
        "target",
        "prediction"
    ).toPandas()

    confusion_matrix = (
        prediction_pdf
        .groupby(["target", "prediction"])
        .size()
        .reset_index(name="count")
    )

    sample_predictions = predictions.select(
        "flower_id",
        "sepal_length",
        "sepal_width",
        "petal_length",
        "petal_width",
        "species",
        "target",
        "prediction"
    ).toPandas()

    with tempfile.TemporaryDirectory() as tmpdir:
        cm_path = os.path.join(tmpdir, f"{model_name}_confusion_matrix.csv")
        sample_path = os.path.join(tmpdir, f"{model_name}_sample_predictions.csv")

        confusion_matrix.to_csv(cm_path, index=False)
        sample_predictions.to_csv(sample_path, index=False)

        mlflow.log_artifact(cm_path)
        mlflow.log_artifact(sample_path)

In [0]:
import os

MLFLOW_DFS_TMP = "/Volumes/gr5069/tj2611/inclass/mlflow_tmp"

os.environ["MLFLOW_DFS_TMP"] = MLFLOW_DFS_TMP
dbutils.fs.mkdirs(MLFLOW_DFS_TMP)

print("MLFLOW_DFS_TMP is set to:", MLFLOW_DFS_TMP)

MLFLOW_DFS_TMP is set to: /Volumes/gr5069/tj2611/inclass/mlflow_tmp


In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="target",
    maxIter=20,
    regParam=0.1,
    elasticNetParam=0.0
)

lr_pipeline = Pipeline(stages=[assembler, scaler, lr])

with mlflow.start_run(run_name="logistic_regression_iris_setosa_prediction_fixed") as run:
    lr_model = lr_pipeline.fit(train_df)
    lr_predictions = lr_model.transform(test_df)

    lr_metrics = evaluate_predictions(lr_predictions)

    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_param("prediction_task", "Predict whether an iris flower is setosa")
    mlflow.log_param("maxIter", 20)
    mlflow.log_param("regParam", 0.1)
    mlflow.log_param("elasticNetParam", 0.0)
    mlflow.log_param("features", ",".join(feature_cols))

    mlflow.log_metric("auc", lr_metrics["auc"])
    mlflow.log_metric("accuracy", lr_metrics["accuracy"])
    mlflow.log_metric("f1", lr_metrics["f1"])
    mlflow.log_metric("weighted_precision", lr_metrics["weighted_precision"])

    mlflow.spark.log_model(
        lr_model,
        "logistic_regression_model",
        dfs_tmpdir=MLFLOW_DFS_TMP
    )

    log_two_artifacts(lr_predictions, "logistic_regression")

    lr_run_id = run.info.run_id

print("Logistic Regression run id:", lr_run_id)
print(lr_metrics)

2026/04/27 00:24:49 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.2) contains a local version label (+databricks.connect.18.0.2). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/04/27 00:24:51 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-dac45fa4-d68e-43c7-b3fa-29/tmpvg63pil_/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
2026/04/27 00:24:51 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when l

Logistic Regression run id: 79df6692c0da4dd0b9481f1bc718f262
{'auc': 1.0, 'accuracy': 1.0, 'f1': 1.0, 'weighted_precision': 1.0}


In [0]:
lr_output = (
    lr_predictions
    .select(
        "flower_id",
        "sepal_length",
        "sepal_width",
        "petal_length",
        "petal_width",
        "species",
        "target",
        "prediction",
        "probability"
    )
    .withColumn("probability", col("probability").cast("string"))
    .withColumn("model_name", lit("Logistic Regression"))
    .withColumn("run_id", lit(lr_run_id))
    .withColumn("prediction_timestamp", current_timestamp())
)

lr_output.write.mode("overwrite").saveAsTable(
    f"{DATABASE_NAME}.logistic_regression_iris_predictions_hw5"
)

display(spark.table(f"{DATABASE_NAME}.logistic_regression_iris_predictions_hw5"))

flower_id,sepal_length,sepal_width,petal_length,petal_width,species,target,prediction,probability,model_name,run_id,prediction_timestamp
9,4.4,2.9,1.4,0.2,setosa,1,1.0,"[0.19327059355224036,0.8067294064477597]",Logistic Regression,79df6692c0da4dd0b9481f1bc718f262,2026-04-27T00:24:55.270Z
12,4.8,3.4,1.6,0.2,setosa,1,1.0,"[0.14580960173431706,0.8541903982656829]",Logistic Regression,79df6692c0da4dd0b9481f1bc718f262,2026-04-27T00:24:55.270Z
14,4.3,3.0,1.1,0.1,setosa,1,1.0,"[0.1338825114704505,0.8661174885295495]",Logistic Regression,79df6692c0da4dd0b9481f1bc718f262,2026-04-27T00:24:55.270Z
22,5.1,3.7,1.5,0.4,setosa,1,1.0,"[0.13817341801369717,0.8618265819863028]",Logistic Regression,79df6692c0da4dd0b9481f1bc718f262,2026-04-27T00:24:55.270Z
24,5.1,3.3,1.7,0.5,setosa,1,1.0,"[0.2512983767171402,0.7487016232828598]",Logistic Regression,79df6692c0da4dd0b9481f1bc718f262,2026-04-27T00:24:55.270Z
25,4.8,3.4,1.9,0.2,setosa,1,1.0,"[0.16371872604142557,0.8362812739585744]",Logistic Regression,79df6692c0da4dd0b9481f1bc718f262,2026-04-27T00:24:55.270Z
26,5.0,3.0,1.6,0.2,setosa,1,1.0,"[0.2504724249588709,0.7495275750411291]",Logistic Regression,79df6692c0da4dd0b9481f1bc718f262,2026-04-27T00:24:55.270Z
28,5.2,3.5,1.5,0.2,setosa,1,1.0,"[0.15471994628292784,0.8452800537170722]",Logistic Regression,79df6692c0da4dd0b9481f1bc718f262,2026-04-27T00:24:55.270Z
31,4.8,3.1,1.6,0.2,setosa,1,1.0,"[0.2044830487637947,0.7955169512362052]",Logistic Regression,79df6692c0da4dd0b9481f1bc718f262,2026-04-27T00:24:55.270Z
35,4.9,3.1,1.5,0.2,setosa,1,1.0,"[0.20731213010970317,0.7926878698902968]",Logistic Regression,79df6692c0da4dd0b9481f1bc718f262,2026-04-27T00:24:55.270Z


In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="target",
    numTrees=50,
    maxDepth=5,
    seed=42
)

rf_pipeline = Pipeline(stages=[assembler, scaler, rf])

with mlflow.start_run(run_name="random_forest_iris_setosa_prediction_fixed") as run:
    rf_model = rf_pipeline.fit(train_df)
    rf_predictions = rf_model.transform(test_df)

    rf_metrics = evaluate_predictions(rf_predictions)

    mlflow.log_param("model_type", "Random Forest")
    mlflow.log_param("prediction_task", "Predict whether an iris flower is setosa")
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("maxDepth", 5)
    mlflow.log_param("seed", 42)
    mlflow.log_param("features", ",".join(feature_cols))

    mlflow.log_metric("auc", rf_metrics["auc"])
    mlflow.log_metric("accuracy", rf_metrics["accuracy"])
    mlflow.log_metric("f1", rf_metrics["f1"])
    mlflow.log_metric("weighted_precision", rf_metrics["weighted_precision"])

    mlflow.spark.log_model(
        rf_model,
        "random_forest_model",
        dfs_tmpdir=MLFLOW_DFS_TMP
    )

    log_two_artifacts(rf_predictions, "random_forest")

    rf_run_id = run.info.run_id

print("Random Forest run id:", rf_run_id)
print(rf_metrics)

2026/04/27 00:25:13 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.2) contains a local version label (+databricks.connect.18.0.2). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/04/27 00:25:14 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-dac45fa4-d68e-43c7-b3fa-29/tmp5j4i6kxb/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
2026/04/27 00:25:14 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when l

Random Forest run id: fe29ae75716e4c8c854a7537757dbd0d
{'auc': 1.0, 'accuracy': 1.0, 'f1': 1.0, 'weighted_precision': 1.0}


In [0]:
rf_output = (
    rf_predictions
    .select(
        "flower_id",
        "sepal_length",
        "sepal_width",
        "petal_length",
        "petal_width",
        "species",
        "target",
        "prediction",
        "probability"
    )
    .withColumn("probability", col("probability").cast("string"))
    .withColumn("model_name", lit("Random Forest"))
    .withColumn("run_id", lit(rf_run_id))
    .withColumn("prediction_timestamp", current_timestamp())
)

rf_output.write.mode("overwrite").saveAsTable(
    f"{DATABASE_NAME}.random_forest_iris_predictions_hw5"
)

display(spark.table(f"{DATABASE_NAME}.random_forest_iris_predictions_hw5"))

flower_id,sepal_length,sepal_width,petal_length,petal_width,species,target,prediction,probability,model_name,run_id,prediction_timestamp
9,4.4,2.9,1.4,0.2,setosa,1,1.0,"[0.0,1.0]",Random Forest,fe29ae75716e4c8c854a7537757dbd0d,2026-04-27T00:25:20.856Z
12,4.8,3.4,1.6,0.2,setosa,1,1.0,"[0.0,1.0]",Random Forest,fe29ae75716e4c8c854a7537757dbd0d,2026-04-27T00:25:20.856Z
14,4.3,3.0,1.1,0.1,setosa,1,1.0,"[0.0,1.0]",Random Forest,fe29ae75716e4c8c854a7537757dbd0d,2026-04-27T00:25:20.856Z
22,5.1,3.7,1.5,0.4,setosa,1,1.0,"[0.0,1.0]",Random Forest,fe29ae75716e4c8c854a7537757dbd0d,2026-04-27T00:25:20.856Z
24,5.1,3.3,1.7,0.5,setosa,1,1.0,"[0.0,1.0]",Random Forest,fe29ae75716e4c8c854a7537757dbd0d,2026-04-27T00:25:20.856Z
25,4.8,3.4,1.9,0.2,setosa,1,1.0,"[0.08,0.92]",Random Forest,fe29ae75716e4c8c854a7537757dbd0d,2026-04-27T00:25:20.856Z
26,5.0,3.0,1.6,0.2,setosa,1,1.0,"[0.0,1.0]",Random Forest,fe29ae75716e4c8c854a7537757dbd0d,2026-04-27T00:25:20.856Z
28,5.2,3.5,1.5,0.2,setosa,1,1.0,"[0.0,1.0]",Random Forest,fe29ae75716e4c8c854a7537757dbd0d,2026-04-27T00:25:20.856Z
31,4.8,3.1,1.6,0.2,setosa,1,1.0,"[0.0,1.0]",Random Forest,fe29ae75716e4c8c854a7537757dbd0d,2026-04-27T00:25:20.856Z
35,4.9,3.1,1.5,0.2,setosa,1,1.0,"[0.0,1.0]",Random Forest,fe29ae75716e4c8c854a7537757dbd0d,2026-04-27T00:25:20.856Z


In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS logistic_regression_prediction_count
FROM {DATABASE_NAME}.logistic_regression_iris_predictions_hw5
"""))

display(spark.sql(f"""
SELECT COUNT(*) AS random_forest_prediction_count
FROM {DATABASE_NAME}.random_forest_iris_predictions_hw5
"""))

display(spark.sql(f"""
SHOW TABLES IN {DATABASE_NAME}
"""))

logistic_regression_prediction_count
29


random_forest_prediction_count
29


database,tableName,isTemporary
tj2611,logistic_regression_iris_predictions_hw5,false
tj2611,logistic_regression_predictions_hw5,false
tj2611,random_forest_iris_predictions_hw5,false
tj2611,random_forest_predictions_hw5,false
